In [1]:
from pydantic import BaseModel
from enum import Enum
from typing import List, Optional, Union
import json
import os
from typing import Optional
from pprint import pprint
from autogen_core.models import UserMessage, SystemMessage, AssistantMessage
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv()

class AgentEnum(str, Enum):
    FlightBooking = "flight_booking"
    HotelBooking = "hotel_booking"
    CarRental = "car_rental"
    ActivitiesBooking = "activities_booking"
    DestinationInfo = "destination_info"
    DefaultAgent = "default_agent"
    GroupChatManager = "group_chat_manager"

# 旅行子任务模型


class TravelSubTask(BaseModel):
    task_details: str
    assigned_agent: AgentEnum  # 我们希望将任务分配给代理


class TravelPlan(BaseModel):
    main_task: str
    subtasks: List[TravelSubTask]
    is_greeting: bool

In [2]:
client = AzureAIChatCompletionClient(
    model=os.getenv("MODEL_FREE_8B"),
    endpoint=os.getenv("API_URL"),
    credential=AzureKeyCredential(os.getenv("API_KEY")),
    model_info={
        "json_output": False,
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

/home/dsa/workspace/ai-agents-for-beginners/venv/lib/python3.12/site-packages/autogen_ext/models/azure/_azure_ai_client.py:307: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(config["model_info"])


In [3]:
# 定义用户消息
messages = [
    SystemMessage(content="""你是一个规划代理。
    你的工作是根据用户的请求决定运行哪些代理。
                      请以JSON格式提供响应，结构如下：
{'main_task': '为一家从新加坡到墨尔本的家庭旅行做计划。',
 'subtasks': [{'assigned_agent': 'flight_booking',
               'task_details': '预订从新加坡到墨尔本的往返航班。'}
    以下是可用于不同任务的专业代理：
    - FlightBooking: 用于预订航班和提供航班信息
    - HotelBooking: 用于预订酒店和提供酒店信息
    - CarRental: 用于预订汽车和提供汽车租赁信息
    - ActivitiesBooking: 用于预订活动和提供活动信息
    - DestinationInfo: 用于提供有关目的地的信息
    - DefaultAgent: 用于处理一般请求""", source="system"),
    UserMessage(
        content="为一个有2个孩子的家庭创建从新加坡到墨尔本的旅行计划", source="user"),
]

response = await client.create(messages=messages, extra_create_args={"response_format": 'json_object'})


response_content: Optional[str] = response.content if isinstance(
    response.content, str) else None
if response_content is None:
    raise ValueError("响应内容不是有效的JSON字符串" )

pprint(json.loads(response_content))

# # 确保响应内容是有效的JSON字符串后再加载
# response_content: Optional[str] = response.content if isinstance(
#     response.content, str) else None
# if response_content is None:
#     raise ValueError("响应内容不是有效的JSON字符串")

# # 加载为JSON后打印响应内容
# pprint(json.loads(response_content))

# 使用MathReasoning模型验证响应内容
# TravelPlan.model_validate(json.loads(response_content))

{'main_task': '为一家从新加坡到墨尔本的家庭旅行做计划。',
 'subtasks': [{'assigned_agent': 'flight_booking',
               'task_details': '预订从新加坡到墨尔本的往返航班，考虑家庭成员（包括两个孩子）的需求。'},
              {'assigned_agent': 'destination_info',
               'task_details': '提供墨尔本的相关信息，如天气情况、适合家庭的景点和活动建议。'},
              {'assigned_agent': 'hotel_booking',
               'task_details': '预订适合家庭入住的酒店，考虑儿童友好的设施和便利的位置。'},
              {'assigned_agent': 'car_rental',
               'task_details': '根据家庭需要，提供墨尔本的汽车租赁建议和预订选项。'},
              {'assigned_agent': 'activities_booking',
               'task_details': '预订适合孩子的活动和景点，例如动物园、博物馆、主题公园等。'}]}
